In [2]:
from datasets import load_dataset

In [4]:
ds.save_to_disk("company-document-dataset-v2-all.parquet")

Saving the dataset (4/4 shards): 100%|██████████| 46969/46969 [00:04<00:00, 10522.60 examples/s]


In [13]:
import os
import json

os.makedirs("../data/company-document-dataset-v2", exist_ok = True)

splits = [
        "train", 
        "validation", 
        "test"
    ]

doctypes = [
        "invoice",
        "packing_slip",
        "purchase_order",
        "quote",
        "shipping_order",
        "work_order"
    ]

ds = load_dataset("company-document-dataset-v2-all.parquet", streaming = True)


for split in splits:
    spl = ds[split]
    os.makedirs(f"../data/company-document-dataset-v2/{split}", exist_ok = True)

    for doctype in doctypes:
        os.makedirs(f"../data/company-document-dataset-v2/{split}/{doctype}", exist_ok = True)
        ds_filtered = spl.filter(lambda x: x["document_type"] == doctype)

        for row in ds_filtered:
            
            pdf_obj = row["pdf"]

            file_name = row.get("file_name").split("/")[-1] or f"document_{i}.pdf"
            file_path = f"../data/company-document-dataset-v2/{split}/{doctype}/{file_name}"

            row["file_name"] = file_name
            row["file_path"] = file_path

            # remove pdf from row to avoid serializing it
            row.pop("pdf", None)

            with open(f"../data/company-document-dataset-v2/{split}/{doctype}/{file_name.split('.pdf')[0]}.json", "w") as f:
                json.dump(row, f)

            if isinstance(pdf_obj, dict):
                with open(file_path, "wb") as f:
                    f.write(pdf_obj["bytes"])
            elif hasattr(pdf_obj, "stream"):
                pdf_obj.stream.seek(0)
                with open(file_path, "wb") as f:
                    f.write(pdf_obj.stream.read())
            elif isinstance(pdf_obj, bytes):
                with open(file_path, "wb") as f:
                    f.write(pdf_obj)
            else:
                raise ValueError(f"Unsupported PDF object type: {type(pdf_obj)}")
